In [ ]:
import pandas as pd

df = pd.read_excel('property/mo_exp.xlsx', index_col=0)



df.columns

df['seq'] = df['seq'].str.replace(' ', '')

df = df.reset_index(drop=True)

df

df[df['PDB ID']=='5KKH']

df[df['short_name']=='NaN']

df['opsin name'] = df['display_name']

# For each row, if short_name is NaN, replace it with '**' + that row's 'opsin name' + '**'
df.loc[df['short_name'].isna(), 'short_name'] = df.loc[df['short_name'].isna(), 'opsin name']

# Do the same for display_name
df.loc[df['display_name'].isna(), 'display_name'] = df.loc[df['display_name'].isna(), 'opsin name']

df

# Check for duplicates in 'opsin name' column
duplicates = df[df.duplicated(subset=['opsin name'], keep=False)]
if not duplicates.empty:
    print("Found duplicates in 'opsin name' column:")
    print(duplicates[['opsin name']])
else:
    print("No duplicates found in 'opsin name' column")

# Check for special characters in 'short_name'
# Define a regular expression that matches any character that is not a letter, number, or underscore
import re
special_char_pattern = r'[^\w]|[()]|\.'  # This matches any non-word character, parentheses, or period
                                         # Note: \w matches letters, numbers, and underscore

# Find rows where short_name contains special characters (except underscores)
special_chars = df[df['short_name'].notna() & df['short_name'].str.contains(special_char_pattern, regex=True)]

if not special_chars.empty:
    print("\nFound special characters in 'short_name' column:")
    for idx, row in special_chars.iterrows():
        print(f"Index: {idx}, short_name: '{row['short_name']}', opsin name: '{row['opsin name']}'")
else:
    print("\nNo special characters found in 'short_name' column")

df['seq'] = df['seq'].fillna(df['sequence'])

df[df['opsin name'] == 'AaClR']['seq']

len(df)

df

# Replace special characters with underscores in both columns
import re

# Define pattern to match any special character except underscore
pattern = r'[^\w]|[()]|\.'  # Matches non-word chars, parentheses, periods (but \w includes underscore)

# Function to replace special chars with underscore while preserving existing underscores
def replace_special_chars(text):
    if pd.isna(text):  # Handle NaN values
        return text
    return re.sub(pattern, '_', str(text))

# Apply the function to both columns
df['short_name'] = df['short_name'].apply(replace_special_chars)
df['display_name'] = df['display_name'].apply(replace_special_chars)

# Print some examples to verify the changes
print("Sample of updated rows:")
sample_rows = df[['opsin name', 'short_name', 'display_name']].head(5)
print(sample_rows)

df[df['seq'].isna()]








import yaml

# Load the YAML file
with open('yaml_configs/test.yaml', 'r') as file:
    config = yaml.safe_load(file)

# Print the loaded configuration to verify
print(config)

# Assuming your YAML data is stored in a variable called 'config'
# and your dataframe is called 'df'

# Get the first entry from the 'seq' column
first_seq = df['seq'].iloc[0]

# Remove any whitespaces from the sequence (as per your earlier requirement)
first_seq = first_seq.replace(' ', '')

# Replace the sequence in the YAML data
config['sequences'][0]['protein']['sequence'] = first_seq

# Print the updated YAML data to verify
print(config)

# Import required libraries
from ruamel.yaml import YAML
import os

# Create the directory if it doesn't exist
os.makedirs('yaml_configs/mo_folding6', exist_ok=True)

# Initialize YAML writer
yaml = YAML()
yaml.indent(mapping=2, sequence=4, offset=2)
yaml.preserve_quotes = True
yaml.width = 1000  # To prevent line wrapping

# Loop through each row in the dataframe
for index, row in df.iterrows():
    # Get the short_name for the filename
    short_name = row['short_name']

    if isinstance(row['seq'], float):
        print("Missing sequence ('seq' column) for entry:", row['short_name'])
        print("Skipping..")
    else:
    # Clean the sequence by removing any whitespaces
        sequence = row['seq'].replace(' ', '')
        
        # Create the YAML structure
        yaml_data = {
            'version': 1,
            'sequences': [
                {
                    'protein': {
                        'id': 'A',
                        'sequence': sequence
                    }
                },
                {
                    'ligand': {
                        'id': 'B',
                        'smiles': 'CC=C(C)C=CC=C(C)C=CC1=C(CCCC1(C)C)C'
                    }
                }
            ]
        }
        
        # Create filename based on short_name
        filename = f'yaml_configs/mo_folding6/{short_name}.yaml'
        
        # Write the file with proper formatting
        with open(filename, 'w') as file:
            yaml.dump(yaml_data, file)
        
        # Add the comment after "version: 1"
        with open(filename, 'r') as file:
            content = file.read()
        
        content = content.replace('version: 1', 'version: 1  # Optional, defaults to 1')
        
        with open(filename, 'w') as file:
            file.write(content)
        
        print(f"Created YAML file: {filename}")

print(f"Finished creating {len(df)} YAML files in yaml_configs/mo_folding6/")

df.to_csv('property/mo_exp.csv')

df[df['short_name']=='Tara_RRB']

# Import required libraries
from ruamel.yaml import YAML
import os

# Create the directory if it doesn't exist
os.makedirs('yaml_configs/mo_folding6', exist_ok=True)

# Initialize YAML writer
yaml = YAML()
yaml.indent(mapping=2, sequence=4, offset=2)
yaml.preserve_quotes = True
yaml.width = 1000  # To prevent line wrapping

# Get the row for Tara_RRB
tara_row = df[df['short_name'] == 'Tara_RRB']

if not tara_row.empty:
    # Get the sequence and clean it
    sequence = tara_row['seq'].iloc[0].replace(' ', '')
    
    # Create the special YAML structure with two ligands
    yaml_data = {
        'version': 1,
        'sequences': [
            {
                'protein': {
                    'id': 'A',
                    'sequence': sequence
                }
            },
            {
                'ligand': {
                    'id': 'B',
                    'smiles': 'CC=C(C)C=CC=C(C)C=CC1=C(CCCC1(C)C)C'
                }
            },
            {
                'ligand': {
                    'id': 'C',
                    'smiles': 'CC=C(C)C=CC=C(C)C=CC1=C(CCCC1(C)C)C'
                }
            }
        ]
    }
    
    # Create filename
    filename = 'yaml_configs/mo_folding6/Tara_RRB_2_retinals.yaml'
    
    # Write the file with proper formatting
    with open(filename, 'w') as file:
        yaml.dump(yaml_data, file)
    
    # Add the comment after "version: 1"
    with open(filename, 'r') as file:
        content = file.read()
    
    content = content.replace('version: 1', 'version: 1  # Optional, defaults to 1')
    
    with open(filename, 'w') as file:
        file.write(content)
    
    print(f"Created special YAML file with two ligands: {filename}")
else:
    print("Error: Could not find 'Tara_RRB' in the dataframe")



# i set it manually to A (was followed by 3 alanines)

# Create a new column 'len' by subtracting 'start' from 'end'
df['len'] = df['end'] - df['start']

# Replace NaN values with 0
df['len'] = df['len'].fillna(0)

# Create a new column for the actual sequence length
df['seq_len'] = df['seq'].str.replace(' ', '').str.len()

# Find entries with length > 30
long_entries = df[df['seq_len'] > 400]

# Print short names of these entries along with both lengths
if not long_entries.empty:
    print(f"Found {len(long_entries)} entries with length > 30:")
    print("FORMAT: short_name: calculated_length (end-start), actual_sequence_length")
    for _, row in long_entries.iterrows():
        print(f"{row['short_name']}: {row['len']}, {row['seq_len']}")
else:
    print("No entries found with length > 30")

df[df['short_name']=='Tara_RRB']

import matplotlib.pyplot as plt
import numpy as np

# Calculate sequence lengths if not already done
if 'seq_len' not in df.columns:
    df['seq_len'] = df['seq'].str.replace(' ', '').str.len()

# Create the histogram
plt.figure(figsize=(10, 6))
plt.hist(df['seq_len'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Distribution of Sequence Lengths')
plt.xlabel('Sequence Length')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)

# Add some stats as text
mean_len = df['seq_len'].mean()
median_len = df['seq_len'].median()
min_len = df['seq_len'].min()
max_len = df['seq_len'].max()

stats_text = f"Mean: {mean_len:.1f}\nMedian: {median_len:.1f}\nMin: {min_len}\nMax: {max_len}"
plt.annotate(stats_text, xy=(0.75, 0.75), xycoords='axes fraction', 
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8))

plt.show()

print("Histogram of sequence lengths has been created and saved as 'opsin_output/figures/sequence_length_histogram.png'")

